## NLP Study

### Bigram Language Model

In this first part, we are going to build a birgram language model. A language model predicts upcoming words.

A bigram model is a special case of a N-gram model for N=2. We utilize the chain rule to calculate the probability of a certain word

$P(w_1, \ldots, w_n) = P(w_1) P(w_2 | w_1) P(w_3 | w_1 w_2) \ldots P(w_n | w_1 \ldots w_{n-1})$

We need some way of simplifying this, in particular $w_1 \ldots w_{n-1}$ could be an arbitrary uncommon creative sentence, such that finding a conditional probability is not very promissing. Therefore we will use the *Markov Assumption* allowing use to approximate

$P(w_n | w_1 \ldots w_{n-1}) \approx P(w_n | w_{w-1})$

such that the above product simplifies to

$P(w_1, \ldots, w_n) = P(w_1) P(w_2 | w_1) P(w_3 | w_2) \ldots P(w_n | w_{n-1})$

We will estimate probabilities using

$\frac{\#(w_{n-1}w_n)}{\#(w_n)}$

and we'll add an extra special symbol $<s>$, at the beginning of the sentence and $</s>$ at the end.


Our training set is a given corpus. The training set should always fit the application of the model. For example if we are trying to create a language recognition model for chemistry lectures, our training dataset should consist of chemistry lectures.

We have three sets, training, dev and test.
We use dev to maybe tune hyperparameters and then use test at the very end to get an unbiased benchmark.

In [19]:
import pandas as pd

def corpus_creator_bigram(text: str, probability: bool = False, add_k_smoothing=0) -> pd.DataFrame:
    words = text.split(" ")
    set_words_list = list(set(words))

    df = pd.DataFrame({w: [0] * len(set_words_list) for w in set_words_list})
    df.index = set_words_list

    for i, word in enumerate(words, start=1):
        if i < len(words):
            df.loc[word, words[i]] += 1


    if add_k_smoothing:
        df.replace(0, add_k_smoothing, inplace=True)

    if probability:
        df = df.apply(lambda x: x / x.sum() if x.sum() > 0 else x, axis=1)

    return df

t_text = """The quick brown fox jumps over the lazy dog. Every morning, the sleepy cat watches silently from the windowsill. Suddenly, a loud bark echoes through the yard, startling the fox. Despite the commotion, the sun rises steadily, warming the dew-covered grass. Later, children laugh and play near the garden, chasing butterflies and kicking a worn-out soccer ball. Meanwhile, an old man sips tea on the porch, reading yesterday’s news. The air smells of fresh earth and blooming lilacs. By evening, the sky blushes with orange and pink hues. Bats flit between trees, and crickets begin their nightly chorus. Peace returns to the little yard, wrapped in shadows and gentle breeze."""
t_text = t_text.replace(".", " .").replace(",", " ,")

df = corpus_creator_bigram(t_text, probability=True, add_k_smoothing=0.001)

import numpy as np

def simple_generate(corpus):
    words = corpus.columns
    generated = []
    while True:
        if generated:
            generated.append(np.random.choice(words, p=corpus.loc[generated[-1]]))
            if generated[-1] == ".":
                break
        else:
            generated =  [np.random.choice([w for w in words if len(w) >= 3])]

    return " ".join(generated).replace(" .", ".").replace(" ,", ",")


simple_generate(df)

'laugh and an old man sips tea on the sleepy cat watches silently from on the porch, reading yesterday’s news.'

In [13]:
test_text = "through the garden, warming the commotion, the sky blushes with orange and gentle breeze."
test_text_two = "orange garden, the jumps"

def calculate_perplexity(text, corpus):
    perp = 1
    word_array = text.replace(".", " .").replace(",", " ,").split(" ")
    for i in range(1, len(word_array)):
        perp *= 1 / corpus.loc[word_array[i-1], word_array[i]]
    return perp ** (1 / (len(word_array) - 1))

[calculate_perplexity(test_text, df), calculate_perplexity(test_text_two, df)]

[2.404340885147961, 87.23354124876344]

Perplexity is a way to evaluate the trained model on the test set (i.e a text). Perplexity is defined as

$P(w_1 \ldots w_N)^{-1/N}$

If there is one word that does not appear in the training dataset but does in the test dataset, we cannot compute the perlexity because we cannot devide by zero. Therefore there are several smoothing algorithms such as laplace smoothing or add k smoothing that just replace the zeros by 1 or k respectively.

There is also interpolation. If we have a trigram and P(w_3 | w_2w_1) is zero we could look at P(w_3 | w_2) or even P(w_3) i.e decrease the context. We can have a look at a weighted average of these values where the weights are lambdas. The lambdas are then parameters which are determined through training with a held out corpus from the training set.

Another way is do do stupid backoff. I.e if the n-gram count is zero, then refer to the n-1-gram count and so on and take the first one that is non-zero. Each time we backoff we discount the probability i.e. by multiplying with a factor lambda < 1. In practice 0.4 works well.

### Embeddings

... (todo)

##### Simple version of word2vec

... (todo)

In [ ]:
import math
import sys
import logging
import re
from tqdm import tqdm

import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("word2vec")

def sigmoid(x):
    if x < 0:
        return np.exp(x)/(1+np.exp(x))
    return 1 / (1 + math.exp(-x))

def loss(w, cpos, cneg):
    return -(np.log(sigmoid(np.dot(cpos, w)))
             + sum(np.log(sigmoid(-np.dot(elem, w))) for elem in cneg))

N = 2  # context window size
V = 8  # embedding vector dimension
k = 2  # number of negatives per context word
epochs = 10  # ... epochs

initial_lr = 0.2

with open("test_text.txt", encoding="utf8") as f:
    t_text = f.read()

wordlist = re.findall("\w+", t_text.lower())
wordsetlist = list(set(wordlist))
noise_weights = [(wordlist.count(word)) ** (0.75) for word in wordsetlist]
noise_weights_sum = sum(noise_weights)
noise_weights = [wei / noise_weights_sum for wei in noise_weights]

W = np.random.rand(len(wordsetlist), V)
C = np.random.rand(len(wordsetlist), V)

word2idx = {w: idx for idx, w in enumerate(wordsetlist)}

for epoch in range(0, epochs):
    logger.info(f"Epoch starting: {epoch}")
    for i, target_word in tqdm(enumerate(wordlist), total=len(wordlist)):
        target_index = word2idx[target_word]

        # Define context window
        start = max(0, i - N)
        end = min(len(wordlist), i + N + 1)
        context_words = [wordlist[j] for j in range(start, end) if j != i]

        for context_word in context_words:
            context_index = word2idx[context_word]

            # Negative sampling
            negative_indices = []
            while len(negative_indices) < k:
                neg_sample = np.random.choice(wordsetlist, p=noise_weights)
                neg_index = word2idx[neg_sample]
                if neg_index != context_index:
                    negative_indices.append(neg_index)

            l_rate = initial_lr * (1 - epoch / epochs)

            # while True:
            C_context_updated = C[context_index, :] - l_rate * (sigmoid(
                np.dot(W[target_index, :], C[context_index, :])
            ) - 1) * W[target_index, :]

            C_negatives_updated = [0] * k

            for j in range(len(negative_indices)):
                C_negatives_updated[j] = C[negative_indices[j], :] - l_rate * (sigmoid(
                    np.dot(W[target_index, :], C[negative_indices[j], :])
                )) * W[target_index, :]

            W_target_updated = W[target_index, :] - l_rate * ((sigmoid(
                np.dot(W[target_index, :], C[context_index, :])
            ) - 1) * C[context_index, :] + sum(sigmoid(np.dot(C[n_index, :], W[target_index, :])) * C[n_index, :]
                        for n_index in negative_indices))

                # if 1.0001 * loss(W[target_index, :], C[context_index, :], [C[l, :] for l in negative_indices]) >= loss(W_target_updated, C_context_updated, C_negatives_updated):
                #     break
                # else:
                #     l_rate /= 2

            W[target_index, :] = W_target_updated
            C[context_index, :] = C_context_updated
            for val, idx in zip(C_negatives_updated, negative_indices):
                C[idx, :] = val

with open('w2f_W.npy', 'wb') as f:
    np.save(f, W)

with open('w2f_C.npy', 'wb') as f:
    np.save(f, C)


2025-05-16 03:02:34,006 [INFO] Epoch starting: 0


100%|██████████| 35163/35163 [05:18<00:00, 110.46it/s]

2025-05-16 03:07:52,342 [INFO] Epoch starting: 1



100%|██████████| 35163/35163 [05:00<00:00, 116.94it/s]

2025-05-16 03:12:53,040 [INFO] Epoch starting: 2


...


100%|██████████| 35163/35163 [18:40<00:00, 31.37it/s] 

2025-05-16 05:39:40,082 [INFO] Epoch starting: 8



100%|██████████| 35163/35163 [08:15<00:00, 71.00it/s]

2025-05-16 05:47:55,328 [INFO] Epoch starting: 9



100%|██████████| 35163/35163 [06:42<00:00, 87.31it/s]


In [ ]:
import numpy as np
import re

def similarity(W, C, word1, word2, wordsetlist):
    word1_index = wordsetlist.index(word1)
    word2_index = wordsetlist.index(word2)
    return (np.dot(W[word1_index, :], W[word2_index])
            / np.linalg.norm(W[word1_index, :]) / np.linalg.norm(W[word2_index]))

with open('data/w2f_W.npy', 'rb') as f:
    W = np.load(f, allow_pickle=True)

with open('data/w2f_C.npy', 'rb') as f:
    C = np.load(f, allow_pickle=True)

with open("test_text.txt", encoding="utf8") as f:
    t_text = f.read()

wordlist = re.findall("\w+", t_text.lower())
wordsetlist = list(set(wordlist))



similarity((W + C)/2, C, "our", "we", wordsetlist)

0.7177318972291924

### Results

(todo)